# INTR on Oxford-IIIT Pet — analysis runner (after training)

Runs `evaluate.py` (accuracy, per-breed results, confusion matrix, training curves, McNemar)
and `attention_metrics.py` (insertion/deletion, attention on the pet, head diversity,
class distinctiveness, test-resolution check, head figures). Output: `analysis.zip`.

**Setup:** Accelerator **GPU T4 x2**, Internet **On**, Kaggle Secret `GITHUB_TOKEN` (as for training).
Then **Add Input → Your Work → Notebooks** and add the *training notebook versions* whose
outputs you want to analyse (R1 and R3 at minimum; R2 and FC when available).
Set the folders in the next cell (use the file browser on the right to copy paths), then *Run All*.
Takes roughly 30–60 minutes on a T4 for all test images.

In [ ]:
# ===================== OPTIONS =====================
# Downloads the trained weights (and R3's predictions) from Hugging Face.
# INTR runs need only their .pth: their test predictions are recreated from the weights.
import os, urllib.request
HF = "https://huggingface.co/Talloo/intr-oxford-pet/resolve/main"
FILES = {"R1": {"intr_pet_R1.pth": "intr_pet_R1.pth"},
         "R2": {"intr_pet_R2.pth": "intr_pet_R2.pth"},
         "R3": {"resnet50_pet_R3.pth": "resnet50_pet_R3.pth", "R3_predictions.csv": "predictions.csv"}}
RUN_DIRS = {}
for run, files in FILES.items():
    d = f"/kaggle/working/in/{run}"; os.makedirs(d, exist_ok=True)
    for remote, local in files.items():
        urllib.request.urlretrieve(f"{HF}/{remote}", f"{d}/{local}")
    RUN_DIRS[run] = d
for k, d in RUN_DIRS.items():
    print(k, sorted(os.listdir(d)))
GITHUB_REPO = "https://github.com/TalalImran07/intr-ml-project.git"
RESOLUTIONS = "320 384 448 512"   # "" to skip the test-resolution check
MAX_IMAGES = None                 # e.g. 200 for a quick test


In [ ]:
import os, shutil, subprocess
CODE = "/kaggle/working/code"
if os.path.exists(CODE):
    shutil.rmtree(CODE)
url = GITHUB_REPO
try:
    from kaggle_secrets import UserSecretsClient
    url = url.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    pass
subprocess.run(["git", "clone", "--depth", "1", url, CODE], check=True)
os.chdir(CODE)
for k, d in RUN_DIRS.items():
    print(k, d, "->", sorted(os.listdir(d)) if os.path.isdir(d) else "NOT FOUND")

In [ ]:
!python prepare_data.py --root /tmp/data

In [ ]:
# Copy every run into /kaggle/working/runs/<name>; re-create predictions for INTR runs that only have weights
import glob
RUNS = "/kaggle/working/runs"
for k, d in RUN_DIRS.items():
    dst = f"{RUNS}/{k}"
    shutil.copytree(d, dst, dirs_exist_ok=True)
    if not os.path.exists(f"{dst}/predictions.csv"):
        w = glob.glob(f"{dst}/intr_pet_*.pth")
        assert w, f"{k}: no predictions.csv and no intr_pet_*.pth in {d}"
        print(f"{k}: re-running the final test evaluation from {os.path.basename(w[0])}")
        !python evaluate_weights.py --weights {w[0]} --data_root /tmp/data --out_dir {dst}
    print(k, sorted(os.listdir(dst)))
RUN_DIRS = {k: f"{RUNS}/{k}" for k in RUN_DIRS}

In [ ]:
runs = " ".join(RUN_DIRS.values())
!python evaluate.py --runs {runs} --data_root /tmp/data --out_dir /kaggle/working/analysis

In [ ]:
import glob
intr = " ".join(glob.glob(f"{d}/intr_pet_*.pth")[0] for k, d in RUN_DIRS.items() if k != "R3")
judge = glob.glob(f"{RUN_DIRS['R3']}/resnet50_pet_*.pth")[0]
cmd = (f"python attention_metrics.py --intr {intr} --judge {judge} --data_root /tmp/data "
       f"--out_dir /kaggle/working/analysis")
if RESOLUTIONS:
    cmd += f" --resolutions {RESOLUTIONS}"
if MAX_IMAGES:
    cmd += f" --max_images {MAX_IMAGES}"
print(cmd)
!{cmd}

In [ ]:
shutil.make_archive("/kaggle/working/analysis", "zip", "/kaggle/working/analysis")
print(sorted(os.listdir("/kaggle/working/analysis")))
print("Download analysis.zip from the Output panel and send it over.")